### Structured Output ::
   
 Structured output means asking an AI to return information in a fixed format instead of ordinary text.

### Pydantic ::

Pydantic is a Python library that defines the structure of your data and validates whether the data follows your rules.

In [2]:
import os
from langchain.chat_models import init_chat_model
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
model=init_chat_model("google_genai:gemini-3.5-flash")


In [3]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released.")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")

model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7', 'langchain-google-genai': '4.4.2'}}, profile={'name': 'Gemini 3.5 Flash', 'release_date': '2026-05-19', 'last_updated': '2026-05-19', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'medium'}, google_api_key=SecretStr('**********'), model='gemini-3.5-flash', temperature=None, client=<google.genai.client.Client object at 0x0000020FE555B380>, default_metadata=(

In [4]:
#Unstructured output
model.invoke("Provide the detail about the movie 3 Idiots")


AIMessage(content=[{'type': 'text', 'text': '***3 Idiots*** is a 2009 Indian Hindi-language coming-of-age comedy-drama film directed by **Rajkumar Hirani** and produced by **Vidhu Vinod Chopra**. Loosely adapted from Chetan Bhagat’s novel *Five Point Someone*, the movie is widely considered one of the greatest and most influential films in Indian cinematic history. \n\nIt achieved massive commercial success both in India and internationally (especially in China, East Asia, and the Middle East) and sparked critical discussions about the pressures of the modern education system.\n\n---\n\n### **1. Basic Information**\n* **Release Date:** December 25, 2009\n* **Director:** Rajkumar Hirani\n* **Screenplay:** Rajkumar Hirani and Abhijat Joshi\n* **Music:** Shantanu Moitra\n* **Runtime:** 170 minutes\n* **Genre:** Comedy / Drama / Coming-of-Age\n\n---\n\n### **2. Main Cast and Characters**\n* **Aamir Khan** as *Ranchoddas "Rancho" Shamaldas Chanchad / Phunsukh Wangdu*: A brilliant, free-spir

In [5]:
model_with_structure.invoke("Provide the detail about the movie 3 Idiots")

Movie(title='3 Idiots', year=2009, director='Rajkumar Hirani', rating=8.4)

### Message output alongside parsed structure

In [6]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    """ Detail about the movie"""
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released.")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")

model_with_structure=model.with_structured_output(Movie,include_raw=True)
response=model_with_structure.invoke("Provide the detail about the movie Inception")
response

{'raw': AIMessage(content=[{'type': 'text', 'text': '{"title":"Inception","year":2010,"director":"Christopher Nolan","rating":8.8}', 'extras': {'signature': 'EooKCocKAWkUfRNM753wjCFUDOps8NJQMOcFbK19X0sdkwpu73KmHEfhUBgPVyx8+GHiAfLkr2nCkqPDLArVkO2J8y/9sH29RHhwDeRIWw8u1B3tXDbIMIcSfyWQHt4yA/Mfb9Zg+A6Qcu/pDtCz9gCj4FReqEtv0RY7ybsco7H5DaJk9QtF7BjMSuQ0YfQG8fDyhJPXaSQlFsDwBLa80bawr+aOVffqaghfGEnULVQi8qe1aWRtriGqct9U67QUHl10ZGkpXuYCGZeoFcGeDw/TZAY1v7dKmfjXYlclDr2hcYHgGSxigUd41q7s6IvzMf2dja0oAQfx+vd1a+oLPakfRlYZXfU6n2i57lIDrUxztnoPhaABGlqCOPJSCswLNJqpaEfkLiYvkDjVEVItnblD16gKvKezfvMkKyJrXA/A1i65sp+b+U4mwrd9sBjrovgGFLuS902rZiA0edvZd/f5xAsckA7ebwPiabYsCfWRth/Bjmsxe0fLp5oy8xqPp+n+OGWK1c4SYWIxlP0DJOmiG4nrBK7Bkt98T3wfiyyDhrdZ7FkVXw5MrEHPq0kXhJCuCvkgMgBHw4jP5tEEWa142/3C+r0ZVHN/eOZsfGsqOsLPah5Jiwxz3tA+Sk+UcWX+wPhhMrH9QcunZfTIy0bqDITqrM8kPtUgh6RC6w5Z1pcsvz6JwvFlkOkenn5X7QDAd2ta5yvEAzRW1vyPHKuGba0TpEq2nygq9BVr+d+LjANO8h1d8Rtmut0BJ8wnVVqwAZuhc53HhF7hOEsiXEcCW+JB6RMsqCAFevGO1nYAvbEymTbe6rLcjONuVRG4ntBDpgqOAE

### Nested Structure

In [7]:
from pydantic import BaseModel,Field
class Actor(BaseModel):
    name:str
    role:str

class MovieDetail(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None=Field(None,description="Budget in million USD")

model_with_structure=model.with_structured_output(MovieDetail)
response=model_with_structure.invoke("Give me the detail about the movie Spiderman")
response


GoogleRateLimitError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 6m41.303969921s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Gen

### TypedDict ::

TypedDict lets you define what keys a dictionary should have and what type of value each key should contain.

In [ ]:
from typing_extensions import TypedDict,Annotated
class MovieDict(TypedDict):
    """A movie with details..."""
    title:Annotated[str,...,"The title of the movie."]
    year:Annotated[int,...,"The year the movie was released."]
    director:Annotated[str,...,"The direcotor of the movie."]
    rating:Annotated[float,...,"The movie's rating out of 10."]

model_withTypedDict=model.with_structured_output(MovieDict)
response=model_withTypedDict.invoke("Please provide the details of the movie Avengers.")
response


{'title': 'The Avengers',
 'year': 2012,
 'director': 'Joss Whedon',
 'rating': 8.0}

### Nested Structure 

In [ ]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetail(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None=Field(None,description="Budget in million USD")

model_with_structure=model.with_structured_output(MovieDetail)
response=model_with_structure.invoke("Give me the detail about the movie Spiderman")
response

{'title': 'Spider-Man',
 'year': 2002,
 'cast': [{'name': 'Tobey Maguire', 'role': 'Peter Parker / Spider-Man'},
  {'name': 'Willem Dafoe', 'role': 'Norman Osborn / Green Goblin'},
  {'name': 'Kirsten Dunst', 'role': 'Mary Jane Watson'},
  {'name': 'James Franco', 'role': 'Harry Osborn'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'budget': 139000000}

In [ ]:
model.profile

{'name': 'Gemini 3.5 Flash',
 'release_date': '2026-05-19',
 'last_updated': '2026-05-19',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True,
 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'],
 'reasoning_effort_default': 'medium'}

### Data Class

A data class is a class typically containing mainly data, althought there aren't really any restrictions.You create it using the @dataclass decorator.

In [ ]:
##Pydantic
import os
from pydantic import BaseModel,Field
from  langchain.agents import create_agent
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

class ContactInfo(BaseModel):
    """Contact information of a person."""
    name:str=Field(description="The name of the person")
    email:str=Field(description="The Email of the person.")
    phone:str=Field(description="The phone no. of the person.")

agent=create_agent(
    model="google_genai:gemini-3.5-flash",
    response_format=ContactInfo
)
result=agent.invoke({
    "messages":[
        {
            "role":"user","content":"Extract contact info from: Divya ,divya@gmail.com,(+91) xxxxxxxxxx"
        }
    ]
})
print(result["structured_response"])


name='Divya' email='divya@gmail.com' phone='(+91) xxxxxxxxxx'


In [18]:
#Typed_Dict
import os
from typing_extensions import TypedDict
from  langchain.agents import create_agent
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

class ContactInfo(BaseModel):
    """Contact information of a person."""
    name:str
    email:str
    phone:str

agent=create_agent(
    model="google_genai:gemini-3.5-flash",
    response_format=ContactInfo
)
result=agent.invoke({
    "messages":[
        {
            "role":"user","content":"Extract contact info from: Divya ,divya@gmail.com,(+91) xxxxxxxxxx"
        }
    ]
})
print(result["structured_response"])


name='Divya' email='divya@gmail.com' phone='(+91) xxxxxxxxxx'


In [19]:
##Data Class
from dataclasses import dataclass
from langchain.agents import create_agent
@dataclass
class ContactInfo:
    """Contact information for a person."""
    name:str
    email:str
    phone:str

result=agent.invoke({
    "messages":[
        {
            "role":"user","content":"Extract contact info from: Divya ,divya@gmail.com,(+91) xxxxxxxxxx"
        }
    ]
})
print(result["structured_response"])

name='Divya' email='divya@gmail.com' phone='(+91) xxxxxxxxxx'
